In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import accuracy_score,  classification_report 
from datasets import load_dataset, concatenate_datasets
from peft import LoraConfig, get_peft_model, TaskType
from collections import Counter
import numpy as np
import os

os.environ['HF_TOKEN'] = "<hf_token>"

# Loading the model from Transformers 

In [ ]:
model_id = "google/gemma-3-4b-it"
print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    device_map="cuda",
    dtype="bfloat16",
  num_labels=2, 
  problem_type="single_label_classification"
)

print("Local model loaded successfully!")
print(f" Parameters: {model.num_parameters():,}")
print(f" Vocab size: {len(tokenizer)}")
print(f" Model size: ~{model.num_parameters() * 2 / 1e9:.1f} GB (bfloat16)")

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Gemma3ForSequenceClassification LOAD REPORT from: google/gemma-3-4b-it
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Local model loaded successfully!
Parameters: 4,300,084,592
Vocab size: 262145
Model size: ~8.6 GB (bfloat16)


## Load the dataset


In [ ]:
path = os.path.dirname(os.path.dirname(os.getcwd()))
data_dir = os.path.join(path, 'data')


def build_prompt(example):
    INSTRUCTION_TEXT = (
        "Classify the speaker in the following example as 'healthy' (control) "
        "or 'patient' in the 'label' column. Answer only with the label."
    )
    question = example.get("question", "") or ""
    answer = example.get("answer", "") or ""

    parts = [INSTRUCTION_TEXT, ""]

    convo = ""
    if question:
        convo += "Psychologist question: " + question + "\n\n"
    convo += "Patient answer: " + answer

    example["text"] = INSTRUCTION_TEXT + "\n\n" + convo
    return example

def load_full_dataset(group:str):
    hc = load_dataset('json', data_files=f'data/ordered_healthy_{group}_dataset.json')["train"]
    pt = load_dataset('json', data_files=f'data/ordered_PT_{group}_dataset.json')["train"]

    hc = hc.map(lambda ex: {**ex, "labels": "healthy", "label": 0})
    pt = pt.map(lambda ex: {**ex, "labels": "patient", "label": 1})
    return concatenate_datasets([hc, pt]).shuffle(seed=42).map(build_prompt)
# Train
train_dataset = load_full_dataset('train')
# Eval
eval_dataset = load_full_dataset('eval')
# Test
test_dataset = load_full_dataset('test')


# OVERSAMPLING
label_counts = Counter(train_dataset["label"])
majority_class = 0 if label_counts[0] > label_counts[1] else 1
minority_class = 1 - majority_class
majority_count = max(label_counts[0], label_counts[1])
minority_count = min(label_counts[0], label_counts[1])

print(f"\nOversampling minority class ({minority_class})...")

# Split dataset by class
majority_samples = train_dataset.filter(lambda x: x["label"] == majority_class)
minority_samples = train_dataset.filter(lambda x: x["label"] == minority_class)

# Oversample minority class by repeating samples
oversample_indices = np.random.choice(
    len(minority_samples), 
    size=majority_count - minority_count, 
    replace=True)
oversampled_minority = minority_samples.select(oversample_indices.tolist())

# Concatenate and shuffle
train_dataset = concatenate_datasets([train_dataset, oversampled_minority]).shuffle(seed=42)

label_counts_after = Counter(train_dataset["label"])


columns_to_remove = [col for col in train_dataset.column_names if col not in ["text", "label"]]
if columns_to_remove:
    train_dataset = train_dataset.remove_columns(columns_to_remove)
    eval_dataset = eval_dataset.remove_columns(columns_to_remove)
    test_dataset = test_dataset.remove_columns(columns_to_remove)
print("SFT Dataset loaded:")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Eval samples: {len(eval_dataset)}")
print(f"  Test samples: {len(test_dataset)}")
print(f"\nSingle Sample: {train_dataset[0]}")


Oversampling minority class (0)...
SFT Dataset loaded:
  Train samples: 14156
  Eval samples: 1076
  Test samples: 1017

Single Sample: {'label': 0, 'text': "Classify the speaker in the following example as 'healthy' (control) or 'patient' in the 'label' column. Answer only with the label.\n\nPsychologist question: oh okay .\n we'll move on to the next part then .\n\nPatient answer: Okie Dokie ."}


# Train with LoRA

In [ ]:

GLU_MODULES = ["w1", "w2", "w3"]
MHA_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj"]
CONV_MODULES = ["in_proj", "out_proj"]


lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=32,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=GLU_MODULES + MHA_MODULES + CONV_MODULES,
    bias="none",
    modules_to_save=None,
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

print("LoRA configuration applied!")
print(f" LoRA rank: {lora_config.r}")
print(f" LoRA alpha: {lora_config.lora_alpha}")
print(f" Target modules: {lora_config.target_modules}")

trainable params: 20,780,032 || all params: 4,320,864,624 || trainable%: 0.4809
LoRA configuration applied!
LoRA rank: 32
LoRA alpha: 32
Target modules: {'w1', 'v_proj', 'k_proj', 'in_proj', 'w3', 'w2', 'out_proj', 'q_proj'}


## Launch Training

In [ ]:


def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=False,   
        max_length=512,
        return_token_type_ids=True,
    )

print("Tokenizing datasets...")
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=["text"])
eval_dataset = eval_dataset.map(preprocess_function, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
)

training_args = TrainingArguments(
    output_dir="./clf",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    label_names=["labels"],
    remove_unused_columns=False,
)

print("Creating LoRA trainer...")
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("\n Starting LoRA training...")
trainer.train()

print("LoRA training completed!")
merged_model = lora_model.merge_and_unload()
merged_model.push_to_hub("PabloCano1/gemma3-4b-classifier")
tokenizer.push_to_hub("PabloCano1/gemma3-4b-classifier")
print(f"LoRA model saved")

🔧 Tokenizing datasets...


Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Creating LoRA trainer...

 Starting LoRA training...


Epoch,Training Loss,Validation Loss
1,0.483770,0.608481
2,0.460897,0.941260


LoRA training completed!


...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

LoRA model saved


In [ ]:

# Tokenize test dataset
print("Tokenizing test dataset...")
def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=True,
        max_length=512,
        return_token_type_ids=True,
    )
test_dataset = test_dataset.map(preprocess_function, batched=True,remove_columns=['text'])


Tokenizing test dataset...


In [ ]:
print("Making predictions on test set...")
predictions = trainer.predict(test_dataset)
y_pred = predictions.predictions.argmax(-1)

# Calculate accuracy
accuracy = accuracy_score(test_dataset["label"], y_pred)
print(f"\n[test] accuracy: {accuracy:.3f}\n")

# Show classification report
print(classification_report(test_dataset["label"], y_pred, target_names=["healthy", "patient"], digits=3))

Making predictions on test set...



[test] accuracy: 0.757

              precision    recall  f1-score   support

     healthy      0.486     0.565     0.522       239
     patient      0.859     0.816     0.837       778

    accuracy                          0.757      1017
   macro avg      0.672     0.691     0.680      1017
weighted avg      0.771     0.757     0.763      1017

